In [16]:
import json
from pathlib import Path
from datetime import datetime

from docx import Document
from lxml import etree
from PIL import Image
import win32com.client
from docx.opc.constants import RELATIONSHIP_TYPE as RT


# ============================================================
# Rendering assumptions (explicit + auditable)
# ============================================================

WORD_RENDERING_ASSUMPTIONS = {
    "assumptions_version": "word_rendering_v1.0",
    "viewer_application": "Microsoft Word",
    "platform": "Windows",
    "zoom_percent": 100,
    "page_background_rgb": [255, 255, 255],
    "chart_transparency_resolves_to_page": True,
    "gridlines_considered_decorative": True,
    "distribution_format": "digital_word"
}


# ============================================================
# Namespaces
# ============================================================

NS = {
    "a": "http://schemas.openxmlformats.org/drawingml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    "c": "http://schemas.openxmlformats.org/drawingml/2006/chart",
}


# ============================================================
# Figure enumeration (do NOT change once validated)
# ============================================================

def enumerate_figures(doc: Document):
    """
    Enumerate referenced figures in document order.
    Produces a unified list of:
      - static images
      - Word-rendered figures (charts, embedded Excel)
    """

    figures = []
    figure_index = 0

    root = doc.part._element

    # --- Static images ---
    for blip in root.findall(".//a:blip", namespaces=NS):
        rid = blip.get(f"{{{NS['r']}}}embed")
        if not rid:
            continue

        # rel = doc.part.rels.get(rid)
        # if not rel:
        #     continue

        rel = doc.part.rels.get(rid)
        if not rel or rel.is_external:
            continue


        figure_index += 1
        figures.append({
            "figure_id": f"Figure_{figure_index:02d}",
            "figure_index": figure_index,
            "source": {
                "source_type": "image",
                "origin_part": str(rel.target_part.partname)
            },
            "rendered_view": {},
            "background": {},
            "perception": {},
            "ada_measurements": {}
        })

    # --- Word charts ---
    for chart in root.findall(".//c:chart", namespaces=NS):
        rid = chart.get(f"{{{NS['r']}}}id")
        if not rid:
            continue

        rel = doc.part.rels.get(rid)
        if not rel:
            continue

        figure_index += 1
        figures.append({
            "figure_id": f"Figure_{figure_index:02d}",
            "figure_index": figure_index,
            "source": {
                "source_type": "word_chart",
                "origin_part": str(rel.target_part.partname)
            },
            "rendered_view": {},
            "background": {},
            "perception": {},
            "ada_measurements": {}
        })

    return figures


# ============================================================
# Phase-0 renderer (single-pass, authoritative)
# ============================================================

def run_phase0(docx_path, output_dir=r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\output"):
    docx_path = Path(docx_path).resolve()
    output_dir = Path(output_dir)
    rendered_dir = output_dir / "rendered"
    rendered_dir.mkdir(parents=True, exist_ok=True)

    # --- Load DOCX ---
    doc = Document(docx_path)

    figures = enumerate_figures(doc)
    assert figures, "No figures found in document."

    # --- Map image parts ---
#     image_parts = {
#     str(rel.target_part.partname): rel.target_part
#     for rel in doc.part.rels.values()
#     if not rel.is_external
#     and hasattr(rel, "target_part")
#     and rel.target_part.content_type.startswith("image/")
# }
    image_parts = {
    str(rel.target_part.partname): rel.target_part
    for rel in doc.part.rels.values()
    if rel.reltype == RT.IMAGE and not rel.is_external
}



    # --- Start Word once ---
    word = win32com.client.Dispatch("Word.Application")
    word.Visible = False
    doc_word = word.Documents.Open(str(docx_path))

    # --- Collect Word-renderable figures in render order ---
    word_renderables = []

    for shape in doc_word.InlineShapes:
        if shape.HasChart:
            word_renderables.append(shape.Chart)

    for shape in doc_word.Shapes:
        if shape.HasChart:
            word_renderables.append(shape.Chart)

    chart_figs = [f for f in figures if f["source"]["source_type"] == "word_chart"]
    assert len(word_renderables) == len(chart_figs), (
        f"Chart mismatch: Word reports {len(word_renderables)} charts, "
        f"manifest has {len(chart_figs)}."
    )

    word_cursor = 0
    written_files = set()

    # ========================================================
    # SINGLE-PASS RENDERING LOOP
    # ========================================================
    from collections import Counter

    print(
        "FIGURE SOURCE TYPES:",
        Counter(f["source"]["source_type"] for f in figures)
    )

    
    for fig in figures:
        fig_id = fig["figure_id"]
        src_type = fig["source"]["source_type"]

        # -------- Static image --------
        if src_type == "image":
            origin = fig["source"]["origin_part"]
            assert origin in image_parts, f"{fig_id}: image part not found."

            part = image_parts[origin]
            ext = part.content_type.split("/")[-1]
            out_path = rendered_dir / f"{fig_id}.{ext}"

            assert out_path not in written_files, f"{fig_id}: output collision."
            written_files.add(out_path)

            with open(out_path, "wb") as f:
                f.write(part.blob)

            assert out_path.exists() and out_path.stat().st_size > 0

            fig["rendered_view"] = {
                "image_path": str(out_path),
                "render_method": "direct_image",
                "background_inherits_page": True
            }

        # -------- Word-rendered figure --------
        else:
            assert word_cursor < len(word_renderables), (
                f"{fig_id}: chart cursor overflow."
            )

            chart = word_renderables[word_cursor]
            word_cursor += 1

            out_path = rendered_dir / f"{fig_id}.png"

            try:
                method = export_chart_as_png(chart, word, out_path)

                # SUCCESS path
                assert out_path.exists() and out_path.stat().st_size > 0

                fig["rendered_view"] = {
                    "image_path": str(out_path),
                    "render_method": f"word_native_{method}",
                    "background_inherits_page": True
                }

            except RuntimeError as e:
                # EXPECTED failure path
                fig["rendered_view"] = {
                    "image_path": None,
                    "render_method": "word_unrenderable",
                    "error": str(e),
                    "background_inherits_page": True
                }

        # -------- Optional metadata (safe) --------
        rv = fig["rendered_view"]
        if rv.get("image_path"):
            try:
                img = Image.open(rv["image_path"])
                rv["width_px"] = img.width
                rv["height_px"] = img.height
                rv["dpi"] = img.info.get("dpi", (None, None))[0]
            except Exception:
                rv["width_px"] = None
                rv["height_px"] = None
                rv["dpi"] = None
        else:
            rv["width_px"] = None
            rv["height_px"] = None
            rv["dpi"] = None

    for fig in figures:
        rv = fig["rendered_view"]
        if rv["render_method"] == "word_unrenderable":
           continue

        path = rv.get("image_path")
        assert path and Path(path).exists(), (
            f"{fig['figure_id']}: missing rendered image"
        )

    # --- Shutdown Word ---
    doc_word.Close(False)
    word.Quit()

    # --- Write manifest ---
    manifest = {
        "manifest_version": "figure_manifest_v1.0",
        "document": {
            "filename": docx_path.name,
            "document_type": "word",
            "generated_at": datetime.utcnow().isoformat() + "Z"
        },
        "rendering_assumptions": WORD_RENDERING_ASSUMPTIONS,
        "figures": figures
    }

    manifest_path = output_dir / "figure_manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print(
        f"Phase-0 complete:\n"
        f"  Figures rendered: {len(figures)}\n"
        f"  Output directory: {rendered_dir}"
    )

    return manifest_path

def export_chart_as_png(chart, word, out_path):
    """
    Ultimate Word chart export fallback.

    Strategy:
    1. Try Chart.Export()
    2. Try Parent.CopyPicture()
    3. FINAL fallback: Selection.CopyAsPicture()
    """

    # ----------------------------
    # Attempt 1: Native export
    # ----------------------------
    try:
        chart.Parent.Select()
        chart.Export(str(out_path), "PNG")
        if out_path.exists() and out_path.stat().st_size > 0:
            return "chart_export"
    except Exception:
        pass

    # ----------------------------
    # Attempt 2: Parent.CopyPicture (if available)
    # ----------------------------
    try:
        parent = chart.Parent
        if hasattr(parent, "CopyPicture"):
            parent.Select()
            parent.CopyPicture()
            return _paste_clipboard_to_file(word, out_path, method="parent_copypicture")
    except Exception:
        pass

    # ----------------------------
    # Attempt 3: Selection.CopyAsPicture (ALWAYS works)
    # ----------------------------
    try:
        chart.Parent.Select()
        word.Selection.CopyAsPicture()
        return _paste_clipboard_to_file(word, out_path, method="selection_copyaspicture")
    except Exception as e:
        raise RuntimeError(
            "Chart cannot be exported by any method (Export, CopyPicture, CopyAsPicture)"
        ) from e


def _paste_clipboard_to_file(word, out_path, method):
    """
    Paste clipboard contents into a temp document and save as PNG.
    """
    tmp_doc = word.Documents.Add()
    try:
        time.sleep(0.2)  # allow clipboard to populate
        tmp_doc.Range(0, 0).Paste()

        # Paste always produces an InlineShape
        tmp_doc.InlineShapes[0].SaveAsPicture(str(out_path))

        if not out_path.exists() or out_path.stat().st_size == 0:
            raise RuntimeError("Clipboard paste produced empty image")

        return method

    finally:
        tmp_doc.Close(False)

# ============================================================
# Entry point
# ============================================================

if __name__ == "__main__":
    run_phase0(
        r"\\wsl.localhost\\Ubuntu-24.04\\home\\joe\\work\\NotBic\\Ports\\New-Method\\docs\\BTS_Port-Performance-2026_Annual-Report_DRAFT for BTS_12.5.25_asof_12.10.docx"
    )

FIGURE SOURCE TYPES: Counter({'word_chart': 23, 'image': 18})


C:\Users\scien\AppData\Local\Temp\ipykernel_2760\1802777080.py:276: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z"


Phase-0 complete:
  Figures rendered: 41
  Output directory: \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\output\rendered


In [ ]:
WORD_RENDERING_ASSUMPTIONS = {
    "assumptions_version": "word_rendering_v1.0",
    "viewer_application": "Microsoft Word",
    "platform": "Windows",
    "zoom_percent": 100,
    "page_background_rgb": [255, 255, 255],
    "chart_transparency_resolves_to_page": True,
    "gridlines_considered_decorative": True,
    "distribution_format": "digital_word"
}


build_phase0_manifest_with_images(
    docx_path=  r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\docs\BTS_Port-Performance-2026_Annual-Report_DRAFT for BTS_12.5.25_asof_12.10.docx",
    output_dir=r"\\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\output\rendered"
)


STARTING  STARTING  STARTING  STARTING  STARTING  STARTING 
Extracting image for Figure_01 to \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\output\rendered\Figure_01.png
Extracting image for Figure_02 to \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\output\rendered\Figure_02.png
Extracting image for Figure_03 to \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\output\rendered\Figure_03.png
Extracting image for Figure_04 to \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\output\rendered\Figure_04.png
Extracting image for Figure_05 to \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\output\rendered\Figure_05.png
Extracting image for Figure_06 to \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\output\rendered\Figure_06.png
Extracting image for Figure_07 to \\wsl.localhost\Ubuntu-24.04\home\joe\work\NotBic\Ports\New-Method\output\rendered\Figure_07.png
Extracting image for Fi

C:\Users\scien\AppData\Local\Temp\ipykernel_7800\684967107.py:190: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z"


{'manifest_version': 'figure_manifest_v1.1',
 'document': {'filename': 'BTS_Port-Performance-2026_Annual-Report_DRAFT for BTS_12.5.25_asof_12.10.docx',
  'document_type': 'word',
  'distribution': 'digital',
  'generated_at': '2026-01-02T19:59:06.276066Z'},
 'rendering_assumptions': {'assumptions_version': 'word_rendering_v1.0',
  'viewer_application': 'Microsoft Word',
  'platform': 'Windows',
  'zoom_percent': 100,
  'page_background_rgb': [255, 255, 255],
  'chart_transparency_resolves_to_page': True,
  'gridlines_considered_decorative': True,
  'distribution_format': 'digital_word'},
 'figures': [{'figure_id': 'Figure_01',
   'figure_index': 1,
   'caption': None,
   'page_number': None,
   'source': {'source_type': 'image',
    'origin_part': '/word/media/image1.png',
    'embedded_excel': None,
    'extracted_image': '\\\\wsl.localhost\\Ubuntu-24.04\\home\\joe\\work\\NotBic\\Ports\\New-Method\\output\\rendered\\Figure_01.png'},
   'rendered_view': {'image_path': '\\\\wsl.localhos